# gkmsvm-lite application note figures

Reads result files from `results/` and writes figures to `figures/`. Nothing here
computes results or needs a GPU. Each input comes from one `make` target in
`benchmarks/Makefile`; see `benchmarks/README.md`.

| Panel | Input (`results/`) | Make target |
|---|---|---|
| 1A | `agreement_dsqtl.tsv` | `fig1a` |
| 1B | `timing_train.tsv`, `timing_train_T1.tsv` | `fig1b` |
| 1C | `timing_variants.tsv` | `fig1c` |
| 1D | `dsqtl_summed.tsv` | `fig1d` |
| S1 | `dsqtl_retrained_per_model.tsv`, `dsqtl_per_model.tsv` | `s1`, `fig1d` |
| S2 | `throughput.tsv` | `s2` |
| S3 | `training_memory.tsv`, `training_memory_cpu.tsv` | `s3` |
| S3B | `training_memory_ctcf.tsv` | `s3b` |
| S4 | `completeness.tsv` | `s4` |
| S5 | `dart_task4_*.tsv`, `dart_retrained_task4_*.tsv`, `dart_task5_*.tsv` (and `dart_reference.tsv` here) | `s5` |
| check | `dart_retrained_task5_*.tsv`, `dart_retrained_train_*.tsv` (optional) | `s5-retrained` |

A panel whose input is missing is drawn as a labeled empty box, never with
placeholder numbers. The last cell lists missing inputs; set `STRICT=1` to make
it raise. `make figures QUICK=1` reads `results/quick/` and writes `figures/quick/`.

In [ ]:
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

sys.path.insert(0, str(Path.cwd()))
from _common import read_provenance  # noqa: E402

RESULTS = Path(os.environ.get("RESULTS_DIR", "results"))
FIGURES = Path(os.environ.get("FIGURES_DIR", "figures"))
FIGURES.mkdir(parents=True, exist_ok=True)

# Bioinformatics style: 178 mm double column
plt.rcParams.update(
    {
        "font.family": "sans-serif",
        "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
        "font.size": 7,
        "axes.titlesize": 8,
        "axes.labelsize": 7,
        "xtick.labelsize": 6,
        "ytick.labelsize": 6,
        "legend.fontsize": 6,
        "figure.dpi": 150,
        "savefig.dpi": 300,
        "savefig.bbox": "tight",
        "axes.spines.top": False,
        "axes.spines.right": False,
        "pdf.fonttype": 42,
    }
)
FIG_W = 7.0

C_LSGKM = "#7d7f85"  # LS-GKM C
C_LSGKM1 = "#b4b6bb"  # LS-GKM C, 1 thread
C_MLSGKM = "#8c6d31"  # mLS-GKM (multithreaded LS-GKM fork)
C_MLSGKM1 = "#d2bd8a"  # mLS-GKM, 1 thread
C_CPU = "#4682B4"  # gkmsvm-lite CPU
C_CPU1 = "#a9c6de"  # gkmsvm-lite CPU, 1 thread
C_GPU = "#D64848"  # gkmsvm-lite CUDA
C_DSVM = "#5A9418"  # deltaSVM
C_ISM = "#C99400"  # ISM
C_EXPL = "#6C66B0"  # GkmExplain mutation impact
METHOD_COLORS = {
    "deltaSVM": C_DSVM,
    "deltaSVM.pl": C_LSGKM,
    "ISM": C_ISM,
    "GkmExplain": C_EXPL,
}

MISSING = []


def load(name):
    """Read results/<name> (TSV with a provenance header), or None if absent."""
    path = RESULTS / name
    if not path.exists():
        MISSING.append(name)
        print(f"MISSING {path}")
        return None
    prov = read_provenance(path)
    print(
        f"{name}: commit {prov.get('commit')}, {prov.get('host')}, "
        f"{prov.get('gpu') or prov.get('device', '')}"
    )
    return pd.read_csv(path, sep="\t", comment="#")


def missing_panel(ax, name):
    ax.set_xticks([])
    ax.set_yticks([])
    for s in ax.spines.values():
        s.set_visible(True)
        s.set_linestyle(":")
        s.set_color("0.6")
    ax.text(
        0.5,
        0.5,
        f"missing\nresults/{name}",
        ha="center",
        va="center",
        color="0.45",
        transform=ax.transAxes,
    )


def letter(ax, s, x=-0.18):
    ax.text(
        x, 1.06, s, transform=ax.transAxes, fontsize=10, fontweight="bold", va="bottom"
    )


def save(fig, stem):
    for ext in ("pdf", "png"):
        fig.savefig(FIGURES / f"{stem}.{ext}")
    plt.show()


def load_timing(name):
    """Parse the two-section TSV written by timing_train.py."""
    path = RESULTS / name
    if not path.exists():
        MISSING.append(name)
        print(f"MISSING {path}")
        return None, None, None
    prov = read_provenance(path)
    print(
        f"{name}: commit {prov.get('commit')}, {prov.get('gpu', prov.get('device'))}, "
        f"LS-GKM -T {prov.get('lsgkm_threads')}, "
        f"gkmsvm-lite CPU threads {prov.get('lite_cpu_threads')}"
    )
    sections, current = {}, None
    for line in path.read_text().splitlines():
        if line.startswith("## "):
            current = line[3:].strip().lower()
            sections[current] = []
        elif current and line and not line.startswith("#"):
            sections[current].append(line.split("\t"))
    frames = {}
    for key, rows in sections.items():
        df = pd.DataFrame(rows[1:], columns=rows[0]).replace("NA", np.nan)
        frames[key] = df.apply(pd.to_numeric)
    # Both implementations run CPU code on this many threads
    threads = prov.get("lite_cpu_threads") or prov.get("lsgkm_threads")
    return frames.get("training"), frames.get("inference"), threads

## Figure 1

In [ ]:
agree = load("agreement_dsqtl.tsv")
train, infer, n_threads = load_timing("timing_train.tsv")
train_t1, _, _ = load_timing("timing_train_T1.tsv")
variants = load("timing_variants.tsv")
summed = load("dsqtl_summed.tsv")
# AUPRC of a random ranking (fraction of dsQTLs), for 1D and S1: the dsQTL
# AUPRC files don't record their labels, but agreement_dsqtl.tsv covers the same variants
dsqtl_base = agree["label"].mean() if agree is not None else None

In [ ]:
from matplotlib.lines import Line2D
from matplotlib.ticker import MultipleLocator

fig = plt.figure(figsize=(FIG_W, 5.0))
outer = fig.add_gridspec(3, 1, height_ratios=[1.0, 0.16, 1.55], hspace=0.42)
# Top: A's three scatters and B, as equal squares; middle: the implementation
# legend B and C share; bottom: C and D
top = outer[0].subgridspec(1, 4, width_ratios=[1, 1, 1, 1.3], wspace=0.5)
bottom = outer[2].subgridspec(1, 2, width_ratios=[1.55, 1], wspace=0.42)

# Implementation styles shared by B (lines) and C (bars); light and dashed = 1 thread
t_many = n_threads or 16
IMPL = {
    ("LS-GKM", "many"): (C_LSGKM, "o-", f"LS-GKM, {t_many} threads"),
    ("LS-GKM", 1): (C_LSGKM1, "o--", "LS-GKM, 1 thread"),
    ("mLS-GKM", "many"): (C_MLSGKM, "D-", f"mLS-GKM, {t_many} threads"),
    ("mLS-GKM", 1): (C_MLSGKM1, "D--", "mLS-GKM, 1 thread"),
    ("CPU", "many"): (C_CPU, "s-", f"gkmsvm-lite CPU, {t_many} threads"),
    ("CPU", 1): (C_CPU1, "s--", "gkmsvm-lite CPU, 1 thread"),
    ("GPU", None): (C_GPU, "^-", "gkmsvm-lite GPU"),
}


def sci(x):
    """x as a power of ten for the max |Δ| labels, e.g. 1×10⁻⁵."""
    m, e = f"{x:.0e}".split("e")
    return f"{m}×10$^{{{int(e)}}}$"


# A: per-variant agreement with LS-GKM, first -t 3 model in the dump
kinds = [("ISM", "ISM"), ("GkmExplain", "GkmExplain"), ("deltaSVM.pl", "deltaSVM")]
axA = [fig.add_subplot(top[0, i]) for i in range(3)]
letter(axA[0], "A", x=-0.62)
if agree is None:
    for ax in axA:
        missing_panel(ax, "agreement_dsqtl.tsv")
else:
    prefix = sorted(
        {c.rsplit("_", 2)[0] for c in agree.columns if c.startswith("t3_")}
    )[0]
    for ax, (kind, title) in zip(axA, kinds):
        x = agree[f"{prefix}_{kind}_lsgkm"].to_numpy()
        y = agree[f"{prefix}_{kind}_lite"].to_numpy()
        ax.scatter(
            x, y, s=1, color=METHOD_COLORS[kind], alpha=0.5, lw=0, rasterized=True
        )
        lim = np.abs(np.concatenate([x, y])).max() * 1.05
        ax.plot([-lim, lim], [-lim, lim], color="0.3", lw=0.5, ls="--")
        ax.set_xlim(-lim, lim)
        ax.set_ylim(-lim, lim)
        ax.set_aspect("equal")
        ax.set_title(title, pad=3)
        ax.text(
            0.97,
            0.04,
            f"max |Δ|\n{sci(np.abs(y - x).max())}",
            transform=ax.transAxes,
            ha="right",
            va="bottom",
            fontsize=5.5,
        )
        ax.set_xlabel("LS-GKM")
    axA[0].set_ylabel("gkmsvm-lite")

# B: training time vs N
axB = fig.add_subplot(top[0, 3], box_aspect=1)  # square, like A
letter(axB, "B", x=-0.5)
if train is None:
    missing_panel(axB, "timing_train.tsv")
else:
    runs = [
        (train, "LS-GKM", "many", "t_lsgkm_train"),
        (train, "CPU", "many", "t_lite_cpu_train"),
    ]
    if train_t1 is not None:
        runs += [
            (train_t1, "LS-GKM", 1, "t_lsgkm_train"),
            (train_t1, "CPU", 1, "t_lite_cpu_train"),
        ]
    if train["t_lite_gpu_train"].notna().any():
        runs.append((train, "GPU", None, "t_lite_gpu_train"))
    for df, impl, thr, col in runs:
        color, style, _ = IMPL[impl, thr]
        axB.plot(df["N"], df[col], style, color=color, ms=2.5, lw=1)
    axB.set_xscale("log")
    axB.set_yscale("log")
    axB.set_xlabel("Training sequences")
    axB.set_ylabel("Training time (s)")
    axB.grid(True, which="major", alpha=0.2)

# C: time to score the dsQTL variants. ISM and GkmExplain score each variant
# against the SVs; deltaSVM builds a 10-mer table once per model, then scores
# variants with table lookups. Bars at a fixed pitch, only for runs that exist
axC = fig.add_subplot(bottom[0, 0])
letter(axC, "C", x=-0.28)
if variants is None:
    missing_panel(axC, "timing_variants.tsv")
else:
    v = variants[variants["status"] == "ok"]
    order = ["ISM", "GkmExplain", "deltaSVM", "deltaSVM scoring"]
    ylabels = [
        "ISM",
        "GkmExplain",
        "deltaSVM:\nbuild table\n(once)",
        "deltaSVM:\nscore\nvariants",
    ]
    cpu = v[v["device"] == "cpu"]
    t_max = int(cpu["threads"].max()) if len(cpu) else None
    bars = []  # (implementation, device, CPU threads or None for any, style key)
    for thr in dict.fromkeys([t_max, 1]):
        key = "many" if thr > 1 else 1
        bars += [
            ("LS-GKM", "cpu", thr, ("LS-GKM", key)),
            ("mLS-GKM", "cpu", thr, ("mLS-GKM", key)),
            ("gkmsvm-lite", "cpu", thr, ("CPU", key)),
        ]
    bars.append(("gkmsvm-lite", "cuda", None, ("GPU", None)))

    def fmt(t):
        return (
            f"{t / 60:.1f} min"
            if t >= 60
            else f"{t:.0f} s"
            if t >= 10
            else f"{t:.1f} s"
        )

    y, centers = 0, []
    for i, m in enumerate(order):
        first = y
        for impl, dev, thr, key in bars:
            sub = v[
                (v["method"] == m)
                & (v["implementation"] == impl)
                & (v["device"] == dev)
            ]
            if thr is not None:
                sub = sub[sub["threads"] == thr]
            if sub.empty:
                continue
            r = sub.iloc[0]
            axC.barh(y, r["wall_s"], height=0.8, color=IMPL[key][0])
            txt = f" {fmt(r['wall_s'])}"
            if m == "deltaSVM scoring":  # wall time is process start-up and JIT
                txt += f" ({r['compute_s'] * 1e3:.0f} ms scoring)"
            axC.text(r["wall_s"], y, txt, va="center", fontsize=4.5)
            y += 1
        centers.append((first + y - 1) / 2)
        if m == "GkmExplain":  # per-variant methods above, deltaSVM below
            axC.axhline(y, color="0.6", lw=0.5, ls=":")
        y += 1
    axC.set_xscale("log")
    axC.set_yticks(centers, ylabels)
    axC.tick_params(axis="y", length=0)
    axC.set_ylim(y - 1.4, -0.6)
    n_var = int(v.loc[v["method"] != "deltaSVM", "n_items"].max())
    axC.set_xlabel(f"Wall-clock time (s), {n_var:,} dsQTL variants")
    axC.set_xlim(right=axC.get_xlim()[1] * 4)  # room for the labels

# Legend for B and C, one column per implementation
handles = [
    Line2D([], [], color=c, marker=s[0], ls=s[1:], ms=3, lw=1, label=lab)
    for c, s, lab in IMPL.values()
]
leg_ax = fig.add_subplot(outer[1])
leg_ax.axis("off")
leg_ax.legend(
    handles=handles,
    loc="center",
    ncol=4,
    frameon=False,
    handlelength=2.4,
    columnspacing=1.6,
    fontsize=6,
)

# D: summed dsQTL AUPRC, both kernels
axD = fig.add_subplot(bottom[0, 1])
letter(axD, "D", x=-0.32)
if summed is None:
    missing_panel(axD, "dsqtl_summed.tsv")
else:
    summed["base"] = summed["method"].str.replace(r"-sum.*", "", regex=True)
    order = ["deltaSVM", "deltaSVM.pl", "ISM", "GkmExplain"]
    for params, marker, fill in (("paper_t3", "o", True), ("paper_t2", "D", False)):
        sub = summed[summed["params"] == params].set_index("base")
        if sub.empty:
            continue
        for i, m in enumerate(order):
            if m not in sub.index:
                continue
            c = METHOD_COLORS[m]
            axD.scatter(
                sub.loc[m, "auprc"],
                i,
                marker=marker,
                s=22,
                zorder=3,
                facecolor=c if fill else "white",
                edgecolor=c,
                lw=1.2,
            )
            ref = sub.loc[m, "paper_auprc"]
            if pd.notna(ref):
                axD.scatter(ref, i, marker="|", s=60, color="0.2", lw=0.8, zorder=4)
    axD.set_yticks(range(len(order)))
    axD.set_yticklabels(order)
    axD.tick_params(axis="y", length=0)
    axD.set_ylim(len(order) - 0.5, -0.5)
    axD.xaxis.set_major_locator(MultipleLocator(0.004))
    axD.xaxis.set_minor_locator(MultipleLocator(0.002))
    axD.set_xlabel(
        "dsQTL AUPRC (5 models summed)"
        + (f"\nbase rate {dsqtl_base:.3f}" if dsqtl_base is not None else "")
    )
    handles = [
        Line2D([], [], marker="o", ls="", color="0.35", ms=4, label="-t 3 gkmrbf"),
        Line2D(
            [],
            [],
            marker="D",
            ls="",
            mfc="white",
            mec="0.35",
            ms=3.5,
            label="-t 2 gkm_esttrunc",
        ),
        Line2D([], [], marker="|", ls="", color="0.2", ms=7, label="published"),
    ]
    axD.legend(handles=handles, frameon=False, loc="lower left", handletextpad=0.3)
    axD.grid(True, axis="x", alpha=0.2)

save(fig, "fig1")

In [ ]:
# Numbers quoted in the text
rows = []
if train is not None and train["t_lite_gpu_train"].notna().any():
    r = train.dropna(subset=["t_lsgkm_train", "t_lite_gpu_train"]).iloc[-1]
    rows.append(
        {
            "quantity": f"training speedup, GPU vs LS-GKM {n_threads} threads, "
            f"N={int(r['N'])}",
            "value": r["t_lsgkm_train"] / r["t_lite_gpu_train"],
        }
    )
# CPU speedups compare runs from the same file, so both sides use the same threads
for df, t in ((train, n_threads), (train_t1, 1)):
    if df is None:
        continue
    r = df.dropna(subset=["t_lsgkm_train", "t_lite_cpu_train"]).iloc[-1]
    rows.append(
        {
            "quantity": f"training speedup, CPU vs LS-GKM, both {t} thread(s), "
            f"N={int(r['N'])}",
            "value": r["t_lsgkm_train"] / r["t_lite_cpu_train"],
        }
    )
if infer is not None and infer["t_lite_gpu_infer"].notna().any():
    r = infer.dropna(subset=["t_lsgkm_infer", "t_lite_gpu_infer"]).iloc[-1]
    rows.append(
        {
            "quantity": f"prediction speedup, GPU vs LS-GKM, {int(r['n_sv'])} SVs",
            "value": r["t_lsgkm_infer"] / r["t_lite_gpu_infer"],
        }
    )
if variants is not None:
    v = variants[variants["status"] == "ok"]
    for m in v["method"].unique():
        lite = v[(v["method"] == m) & (v["implementation"] == "gkmsvm-lite")]
        name = "deltaSVM table build" if m == "deltaSVM" else m
        for impl in ("LS-GKM", "mLS-GKM"):
            base = v[(v["method"] == m) & (v["implementation"] == impl)]
            if base.empty:
                continue
            # CPU against the C implementation at the same thread count
            for _, b in base.iterrows():
                cpu = lite[
                    (lite["device"] == "cpu") & (lite["threads"] == b["threads"])
                ]
                if len(cpu):
                    rows.append(
                        {
                            "quantity": f"{name} speedup, gkmsvm-lite CPU vs {impl}, "
                            f"both {int(b['threads'])} thread(s)",
                            "value": b["wall_s"] / cpu["wall_s"].iloc[0],
                        }
                    )
            # GPU against its fastest run
            fastest = base.loc[base["wall_s"].idxmin()]
            gpu = lite[lite["device"] == "cuda"]
            if len(gpu):
                rows.append(
                    {
                        "quantity": f"{name} speedup, gkmsvm-lite GPU vs fastest {impl} "
                        f"({int(fastest['threads'])} thread(s))",
                        "value": fastest["wall_s"] / gpu["wall_s"].iloc[0],
                    }
                )
    mem_cols = [
        "method",
        "implementation",
        "device",
        "threads",
        "peak_rss_mb",
        "gpu_pool_mb",
    ]
    variants[mem_cols].to_csv(
        FIGURES / "table_variant_memory.tsv", sep="\t", index=False
    )
if dsqtl_base is not None:
    rows.append(
        {"quantity": "dsQTL AUPRC base rate (fraction of dsQTLs)", "value": dsqtl_base}
    )
if summed is not None:
    for _, r in summed.iterrows():
        rows.append({"quantity": f"dsQTL AUPRC {r['method']}", "value": r["auprc"]})
numbers = pd.DataFrame(rows)
numbers.to_csv(FIGURES / "text_numbers.tsv", sep="\t", index=False)
numbers

## Supplement

In [ ]:
# S1: gkmsvm-lite-trained models vs the paper's lsgkm models, per negset
retrained = load("dsqtl_retrained_per_model.tsv")
paper = load("dsqtl_per_model.tsv")

fig, ax = plt.subplots(figsize=(3.2, 3.0))
if retrained is None or paper is None:
    missing_panel(ax, "dsqtl_retrained_per_model.tsv")
else:
    pairs = {"l10k6": "paper_t2", "l10k6_rbf": "paper_t3"}
    merged = retrained.assign(paper_params=retrained["params"].map(pairs)).merge(
        paper,
        left_on=["paper_params", "negset", "method"],
        right_on=["params", "negset", "method"],
        suffixes=("_lite", "_lsgkm"),
    )
    for (method, params), g in merged.groupby(["method", "params_lite"]):
        ax.scatter(
            g["auprc_lsgkm"],
            g["auprc_lite"],
            s=14,
            color=METHOD_COLORS[method],
            marker="o" if params.endswith("rbf") else "D",
            label=f"{method}, {'-t 3' if params.endswith('rbf') else '-t 2'}",
        )
    lo = merged[["auprc_lite", "auprc_lsgkm"]].min().min() - 0.002
    hi = merged[["auprc_lite", "auprc_lsgkm"]].max().max() + 0.002
    ax.plot([lo, hi], [lo, hi], color="0.3", lw=0.5, ls="--")
    ax.set_xlabel("AUPRC, LS-GKM-trained model")
    if dsqtl_base is not None:
        ax.text(
            0.97,
            0.04,
            f"base rate {dsqtl_base:.3f}",
            transform=ax.transAxes,
            ha="right",
            va="bottom",
            fontsize=5.5,
        )
    ax.set_ylabel("AUPRC, gkmsvm-lite-trained model")
    ax.legend(frameon=False, fontsize=5)
save(fig, "figS1_retrained")

In [ ]:
# S2: prediction throughput vs number of SVs
tput = load("throughput.tsv")
fig, axes = plt.subplots(1, 2, figsize=(FIG_W, 2.6), sharey=True)
# gkmsvm-lite's default path per kernel (its masked-join rows, a forced
# fallback, aren't drawn)
TITLES = {
    "t2": "-t 2 (fast path: exact l-mer table)",
    "t3": "-t 3 (fast path: l-mer index)",
}
for ax, kernel in zip(axes, ("t2", "t3")):
    ax.set_title(TITLES[kernel])
    if tput is None:
        missing_panel(ax, "throughput.tsv")
        continue
    t = tput[
        (tput["kernel"] == kernel)
        & (tput["status"] == "ok")
        & (tput["path"] != "masked join")
    ]
    for (impl, path, dev), g in t.groupby(["implementation", "path", "device"]):
        g = g.sort_values("n_sv")
        if impl in ("LS-GKM", "mLS-GKM"):
            color, style = (C_LSGKM, "o-") if impl == "LS-GKM" else (C_MLSGKM, "D-")
            label = f"{impl} gkmpredict, {int(g['threads'].iloc[0])} threads"
        else:
            color = C_GPU if dev == "cuda" else C_CPU
            style = "^-" if dev == "cuda" else "s-"
            label = f"gkmsvm-lite {'GPU' if dev == 'cuda' else 'CPU'}, fast path"
        ax.plot(
            g["n_sv"],
            g["seqs_per_sec"],
            style,
            color=color,
            ms=3,
            lw=1,
            label=label,
        )
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlabel("Support vectors")
    ax.grid(True, which="major", alpha=0.2)
axes[0].set_ylabel("Sequences per second (300 bp)")
letter(axes[0], "A")
letter(axes[1], "B", x=-0.08)
# One legend for both panels, above them
handles, labels = axes[0].get_legend_handles_labels()
if handles:
    order = sorted(
        range(len(labels)),
        key=lambda i: (
            not labels[i].startswith("LS-GKM"),
            not labels[i].startswith("mLS-GKM"),
            "GPU" in labels[i],
        ),
    )
    fig.legend(
        [handles[i] for i in order],
        [labels[i] for i in order],
        loc="lower center",
        bbox_to_anchor=(0.5, 1.0),
        ncol=3,
        frameon=False,
        fontsize=6,
    )
save(fig, "figS2_throughput")

In [ ]:
# S3: training time and peak memory by solver
memory = load("training_memory.tsv")
memory_cpu = load("training_memory_cpu.tsv")
if memory is not None and memory_cpu is not None:
    memory = pd.concat([memory, memory_cpu], ignore_index=True)
fig, axes = plt.subplots(1, 2, figsize=(FIG_W, 2.6))
# Legend order: LS-GKM, then gkmsvm-lite solvers, GPU before CPU
SOLVER_STYLE = {
    "lsgkm": ("LS-GKM", C_LSGKM, "D-"),
    "libsvm": ("precomputed Gram", C_GPU, "o-"),
    "smo": ("SMO", C_CPU, "s-"),
    "nystrom": ("Nyström", C_EXPL, "^--"),
}
if memory is None:
    for ax in axes:
        missing_panel(ax, "training_memory.tsv")
else:
    ok = memory[memory["status"] == "ok"]
    groups = ok.groupby(["solver", "device"])
    keys = sorted(
        (k for k in groups.groups if k[0] in SOLVER_STYLE),
        key=lambda k: (list(SOLVER_STYLE).index(k[0]), k[1] == "cpu"),
    )
    for solver, device in keys:
        label, color, style = SOLVER_STYLE[solver]
        g = groups.get_group((solver, device)).sort_values("N")
        # gkmsvm-lite runs on the GPU are filled, on CPU open; LS-GKM is CPU only
        cpu = device == "cpu" and solver != "lsgkm"
        if solver != "lsgkm":
            label = f"{label}, {'CPU' if cpu else 'GPU'}"
        kw = dict(color=color, ms=3, lw=1, label=label, mfc="none" if cpu else color)
        if cpu:
            style = style[0] + ":"
        axes[0].plot(g["N"], g["train_s"], style, **kw)
        axes[1].plot(g["N"], g["peak_rss_mb"] / 1024, style, **kw)
    gram = memory.drop_duplicates("N").sort_values("N")
    axes[1].plot(
        gram["N"], gram["gram_gb"], color="0.6", lw=0.8, ls=":", label="N² × 8 bytes"
    )
    for ax, ylabel in zip(axes, ("Training time (s)", "Peak host memory (GB)")):
        ax.set_xscale("log")
        ax.set_yscale("log")
        ax.set_xlabel("Training sequences")
        ax.set_ylabel(ylabel)
        ax.grid(True, which="major", alpha=0.2)
    fig.legend(
        *axes[1].get_legend_handles_labels(),
        loc="lower center",
        bbox_to_anchor=(0.5, 1.0),
        ncol=4,
        frameon=False,
        fontsize=6,
    )
    nys = ok[ok["solver"] == "nystrom"].sort_values("N")
    if len(nys) and nys["r_vs_exact"].notna().any():
        last = nys.dropna(subset=["r_vs_exact"]).iloc[-1]
        axes[0].text(
            0.03,
            0.97,
            f"Nyström vs exact at N={int(last['N']):,}:\n"
            f"r = {last['r_vs_exact']:.4f}, AUROC {last['test_auroc']:.4f}",
            transform=axes[0].transAxes,
            va="top",
            fontsize=5.5,
        )
letter(axes[0], "A")
letter(axes[1], "B")
save(fig, "figS3_solvers")

In [ ]:
# S3B: large-N scaling on LS-GKM's own design (H1-hESC CTCF, 5k-90k; Lee 2016, Fig. 1)
ctcf = load("training_memory_ctcf.tsv")
fig, axes = plt.subplots(1, 2, figsize=(FIG_W, 2.5))
if ctcf is None:
    for ax in axes:
        missing_panel(ax, "training_memory_ctcf.tsv")
else:
    ok = ctcf[ctcf["status"] == "ok"]
    for run, g in ok.groupby("run"):
        g = g.sort_values("N")
        if run.startswith("lsgkm"):
            cache = g["lsgkm_cache_mb"].iloc[0]
            big = cache >= 1000
            label = (
                f"LS-GKM, {cache / 1000:g} GB cache"
                if big
                else f"LS-GKM, {cache:g} MB cache"
            )
            kw = dict(color=C_LSGKM if big else C_LSGKM1, marker="D", ls="-")
        else:
            name, color, style = SOLVER_STYLE[run]
            dev = "CPU" if g["device"].iloc[0] == "cpu" else "GPU"
            label = f"{name}, {dev}"
            kw = dict(color=color, marker=style[0], ls=style[1:])
        axes[0].plot(g["N"], g["train_s"], ms=3, lw=1, label=label, **kw)
        axes[1].plot(g["N"], g["peak_rss_mb"] / 1024, ms=3, lw=1, label=label, **kw)
    gram = ctcf.drop_duplicates("N").sort_values("N")
    axes[1].plot(
        gram["N"], gram["gram_gb"], color="0.6", lw=0.8, ls=":", label="N² × 8 bytes"
    )
    for ax, ylabel in zip(axes, ("Training time (s)", "Peak host memory (GB)")):
        ax.set_xscale("log")
        ax.set_yscale("log")
        ax.set_xlabel("Training sequences (H1-hESC CTCF)")
        ax.set_ylabel(ylabel)
        ax.grid(True, which="major", alpha=0.2)
    axes[1].legend(frameon=False, fontsize=5.5)
    skipped = ctcf[ctcf["status"] == "skipped"]
    if len(skipped):
        axes[1].text(
            0.03,
            0.97,
            "Gram > 75% of RAM from N = "
            f"{int(skipped['N'].min()):,}: precomputed Gram and Nyström skipped",
            transform=axes[1].transAxes,
            va="top",
            fontsize=5,
        )
letter(axes[0], "A")
letter(axes[1], "B")
save(fig, "figS3B_ctcf_scaling")

In [ ]:
# S4: GkmExplain completeness, sum of attributions = f(x) - b
compl = load("completeness.tsv")
fig, ax = plt.subplots(figsize=(2.8, 2.6))
if compl is None:
    missing_panel(ax, "completeness.tsv")
else:
    colors = {"positive": C_GPU, "negative": C_CPU, "random": "0.6"}
    for group, g in compl.groupby("group"):
        ax.scatter(
            g["score_minus_bias"],
            g["attribution_sum"],
            s=6,
            lw=0,
            color=colors.get(group, "0.3"),
            label=group,
        )
    x = compl["score_minus_bias"]
    ax.plot([x.min(), x.max()], [x.min(), x.max()], color="0.3", lw=0.5, ls="--")
    err = (compl["attribution_sum"] - x).abs().max()
    ax.text(0.04, 0.96, f"max |Δ| = {err:.1e}", transform=ax.transAxes, va="top")
    ax.set_xlabel("f(x) − b")
    ax.set_ylabel("Σ GkmExplain attributions")
    ax.legend(frameon=False, loc="lower right")
save(fig, "figS4_completeness")

In [ ]:
# S5: DART-Eval Tasks 4 and 5 (Patel et al. 2024), ENCODE gkm-SVM models and, for
# Task 4, models retrained on DART-Eval's training chromosomes (dart_train.py), against
# the published DNALM, ChromBPNet and Evo 2 results (dart_reference.tsv, with sources)
from matplotlib.lines import Line2D

ref = pd.read_csv("dart_reference.tsv", sep="\t", comment="#")
CELLS = ["GM12878", "H1ESC", "HEPG2", "IMR90", "K562"]
DATASETS = ["caQTL", "dsQTL"]
# ENCODE models scored on Task 5 (both GM12878); Task 4 uses each cell line's DNase model
TASK5_ASSAY = {"ENCFF701NCI": "DNase", "ENCFF953DDE": "ATAC"}
C_EVO2 = "#2E9C9C"
C_RETRAIN = "#E07B39"


def load_dart(pattern):
    files = sorted(RESULTS.glob(pattern))
    if not files:
        MISSING.append(pattern)
        print(f"MISSING {RESULTS / pattern}")
        return None
    return pd.concat(
        [load(f.name).assign(model=f.stem.rsplit("_", 1)[-1]) for f in files],
        ignore_index=True,
    )


task4 = load_dart("dart_task4_*_ENCFF*.tsv")
task5 = load_dart("dart_task5_ENCFF*.tsv")
task4_retrained = load_dart("dart_retrained_task4_*.tsv")

# (setting in dart_reference.tsv, legend label, marker style); x offsets within a category
GROUPS = [
    ("Zero-shot", "DNALMs, zero-shot", dict(marker="x", color="0.55", s=10, lw=0.8)),
    ("Evo 2", "Evo 2 7B / 40B, zero-shot", dict(marker="v", color=C_EVO2, s=16, lw=0)),
    (
        "Probed",
        "DNALMs, probed",
        dict(marker="o", facecolor="white", edgecolor="0.55", s=12, lw=0.7),
    ),
    ("Fine-Tuned", "DNALMs, fine-tuned", dict(marker="o", color="0.55", s=12, lw=0)),
    ("Ab initio", "ChromBPNet", dict(marker="D", color="0.1", s=16, lw=0)),
    ("lite", "gkm-SVM, ENCODE DNase model", dict(marker="s", color=C_CPU, s=20, lw=0)),
    (
        "lite ATAC",
        "gkm-SVM, ENCODE ATAC model (Task 5)",
        dict(marker="s", color=C_CPU1, s=20, lw=0),
    ),
    (
        "retrained",
        "gkm-SVM, retrained (Task 4)",
        dict(marker="s", color=C_RETRAIN, s=20, lw=0),
    ),
]


def ref_values(task, dataset, setting, metric):
    r = ref[
        (ref["task"] == task) & (ref["dataset"] == dataset) & (ref["metric"] == metric)
    ]
    if setting == "Zero-shot":
        r = r[
            r["setting"].str.startswith("Zero-shot")
            & ~r["model"].str.startswith("Evo 2")
        ]
    elif setting == "Evo 2":
        r = r[r["model"].str.startswith("Evo 2")]
    else:
        r = r[r["setting"] == setting]
    return r["value"].to_numpy()


def strip(ax, cats, values_of, groups):
    """One column of points per group within each category."""
    offs = np.linspace(-0.3, 0.3, len(groups))
    for (setting, _, style), off in zip(groups, offs):
        for i, cat in enumerate(cats):
            vals = values_of(cat, setting)
            if len(vals):
                ax.scatter(np.full(len(vals), i + off), vals, zorder=3, **style)
    ax.set_xticks(range(len(cats)), cats)
    ax.set_xlim(-0.6, len(cats) - 0.4)
    ax.grid(True, axis="y", alpha=0.2)


BASE_STYLE = dict(color="0.4", lw=0.7, ls="--")


def base_rate(ax, cats, rows_of, n_pos, n_neg):
    """AUPRC of a random ranking (fraction of positives) in each category."""
    for i, cat in enumerate(cats):
        rows = rows_of(cat)
        for r in (rows[n_pos] / (rows[n_pos] + rows[n_neg])).unique():
            ax.hlines(r, i - 0.4, i + 0.4, zorder=2, **BASE_STYLE)


fig = plt.figure(figsize=(FIG_W, 4.4))
gs = fig.add_gridspec(2, 12, hspace=0.6, wspace=1.6, top=0.9)
T4 = [
    ("auroc", "AUROC"),
    ("auprc", "AUPRC"),
    ("spearman_peaks", "Spearman r, peaks"),
    ("spearman_all", "Spearman r, all regions"),
]
T5 = [("auroc", "AUROC"), ("auprc", "AUPRC"), ("pearson", "Pearson r, effect size")]
axes4 = [fig.add_subplot(gs[0, 3 * i : 3 * i + 3]) for i in range(4)]
axes5 = [fig.add_subplot(gs[1, 4 * i : 4 * i + 4]) for i in range(3)]
groups4 = [
    g
    for g in GROUPS
    if g[0] in ("Probed", "Fine-Tuned", "Ab initio", "lite", "retrained")
]

for ax, (metric, title) in zip(axes4, T4):
    ax.set_title(title, fontsize=7)
    if task4 is None:
        missing_panel(ax, "dart_task4_*.tsv")
        continue

    def values4(cell, setting, metric=metric):
        if setting not in ("lite", "retrained"):
            return ref_values("task4", cell, setting, metric)
        df = task4 if setting == "lite" else task4_retrained
        if df is None or metric not in df:
            return []
        return df.loc[df["cell_line"] == cell, metric].dropna().to_numpy()

    strip(ax, CELLS, values4, groups4)
    if metric == "auprc":
        base_rate(
            ax,
            CELLS,
            lambda c: task4[task4["cell_line"] == c],
            "n_idr_peaks",
            "n_nonpeaks",
        )
    ax.tick_params(axis="x", labelrotation=45, labelsize=5.5)
letter(axes4[0], "A", x=-0.45)

LITE5 = {"pearson": "pearson_effect", "spearman": "spearman_effect"}
for ax, (metric, title) in zip(axes5, T5):
    ax.set_title(title, fontsize=7)
    if task5 is None:
        missing_panel(ax, "dart_task5_*.tsv")
        continue

    def values5(ds, setting, metric=metric):
        if setting.startswith("lite"):
            assay = "ATAC" if setting.endswith("ATAC") else "DNase"
            rows = task5[
                (task5["task"] == ds) & (task5["model"].map(TASK5_ASSAY) == assay)
            ]
            return rows[LITE5.get(metric, metric)].dropna().to_numpy()
        return ref_values("task5", ds, setting, metric)

    # Zero-shot results are AUROC only; retrained models are Task 4 only
    strip(
        ax,
        DATASETS,
        values5,
        [
            g
            for g in GROUPS
            if g[0] != "retrained"
            and (metric == "auroc" or g[0] not in ("Zero-shot", "Evo 2"))
        ],
    )
    if metric == "auprc":
        base_rate(
            ax, DATASETS, lambda ds: task5[task5["task"] == ds], "n_sig", "n_ctrl"
        )
letter(axes5[0], "B", x=-0.3)

handles = [
    Line2D(
        [],
        [],
        ls="",
        marker=st["marker"],
        ms=4,
        color=st.get("color", st.get("edgecolor")),
        mfc=st.get("facecolor", st.get("color")),
        mec=st.get("edgecolor", st.get("color")),
        label=label,
    )
    for _, label, st in GROUPS
]
handles.append(Line2D([], [], label="AUPRC base rate", **BASE_STYLE))
fig.legend(
    handles=handles,
    loc="lower center",
    bbox_to_anchor=(0.5, 0.99),
    ncol=3,
    frameon=False,
    fontsize=6,
    handletextpad=0.3,
    columnspacing=1.2,
)
save(fig, "figS5_dart_eval")

In [ ]:
# Check, not in the paper: the retrained models (dart_train.py) against S5's ENCODE
# models on Task 4 (also in S5) and Task 5 (`make s5-retrained`), with training
# times. Task 5 and training results are optional, so absent ones aren't MISSING
def load_optional(pattern):
    files = sorted(RESULTS.glob(pattern))
    return (
        pd.concat([load(f.name) for f in files], ignore_index=True) if files else None
    )


r_train = load_optional("dart_retrained_train_*.tsv")
r4 = task4_retrained
r5 = load_optional("dart_retrained_task5_*.tsv")
STYLE = {setting: style for setting, _, style in GROUPS}
CHECK = [
    ("Fine-Tuned", "DNALMs, fine-tuned", STYLE["Fine-Tuned"]),
    ("Ab initio", "ChromBPNet", STYLE["Ab initio"]),
    ("lite", "gkm-SVM, ENCODE DNase model", STYLE["lite"]),
    ("retrained", "gkm-SVM, retrained", STYLE["retrained"]),
]


def check_values(df_of, ref_task, key):
    """values_of for strip(): our results from df_of(setting), else published ones."""

    def values(cat, setting, metric):
        if setting not in ("lite", "retrained"):
            return ref_values(ref_task, cat, setting, metric)
        df = df_of(setting)
        if df is None:
            return []
        return df.loc[df[key] == cat, LITE5.get(metric, metric)].dropna().to_numpy()

    return values


encode5 = None if task5 is None else task5[task5["model"].map(TASK5_ASSAY) == "DNase"]
values4 = check_values(
    lambda s: r4 if s == "retrained" else task4, "task4", "cell_line"
)
values5 = check_values(lambda s: r5 if s == "retrained" else encode5, "task5", "task")

if r4 is None and r5 is None:
    print("No retrained results: run `make -C benchmarks s5-retrained`")
else:
    fig = plt.figure(figsize=(FIG_W, 4.4))
    gs = fig.add_gridspec(2, 12, hspace=0.6, wspace=1.6, top=0.9)
    for i, (metric, title) in enumerate(T4):
        ax = fig.add_subplot(gs[0, 3 * i : 3 * i + 3])
        ax.set_title(f"Task 4: {title}", fontsize=7)
        strip(ax, CELLS, lambda c, s: values4(c, s, metric), CHECK)
        ax.tick_params(axis="x", labelrotation=45, labelsize=5.5)
        if metric == "auprc" and r4 is not None:
            base_rate(
                ax,
                CELLS,
                lambda c: r4[r4["cell_line"] == c],
                "n_idr_peaks",
                "n_nonpeaks",
            )
    for i, (metric, title) in enumerate(T5):
        ax = fig.add_subplot(gs[1, 4 * i : 4 * i + 4])
        ax.set_title(f"Task 5: {title}", fontsize=7)
        strip(ax, DATASETS, lambda d, s: values5(d, s, metric), CHECK)
        if metric == "auprc" and r5 is not None:
            base_rate(ax, DATASETS, lambda d: r5[r5["task"] == d], "n_sig", "n_ctrl")
    handles = [
        Line2D([], [], ls="", marker=st["marker"], ms=4, color=st["color"], label=label)
        for _, label, st in CHECK
    ]
    handles.append(Line2D([], [], label="AUPRC base rate", **BASE_STYLE))
    fig.legend(
        handles=handles,
        loc="lower center",
        bbox_to_anchor=(0.5, 0.99),
        ncol=5,
        frameon=False,
        fontsize=6,
        handletextpad=0.3,
        columnspacing=1.2,
    )
    save(fig, "check_dart_retrained")

    # ENCODE vs retrained, side by side
    cols4 = ["auroc", "auprc", "spearman_peaks", "spearman_all"]
    if r4 is not None and task4 is not None:
        display(
            task4[["cell_line"] + cols4]
            .merge(
                r4[["cell_line"] + cols4],
                on="cell_line",
                suffixes=("_encode", "_retrained"),
            )
            .round(3)
        )
    cols5 = ["auroc", "auprc", "pearson_effect"]
    if r5 is not None and encode5 is not None:
        display(
            encode5[["task"] + cols5]
            .merge(r5[["task"] + cols5], on="task", suffixes=("_encode", "_retrained"))
            .round(3)
        )
    if r_train is not None:
        display(r_train)

In [ ]:
if MISSING:
    print("Missing inputs:\n  " + "\n  ".join(MISSING))
    if os.environ.get("STRICT") == "1":
        raise FileNotFoundError(f"{len(MISSING)} missing result files")
else:
    print("All inputs present.")